# 🎙️ Kikuyu Voice Clone — XTTS v2 Fine-Tuning

This notebook fine-tunes Coqui XTTS v2 on your Kikuyu voice recordings.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your ZIP file when prompted in Step 2

**What to ZIP and upload:**
- `public/audio/chunks/` folder (all your WAV files)
- `coqui-server/dataset/metadata.csv`
- `public/audio/voice-training-1.wav`

ZIP them as `kikuyu_data.zip`

In [ ]:
# Step 1 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Step 2 — Upload your data ZIP
from google.colab import files
import zipfile, os

print('Upload kikuyu_data.zip now...')
uploaded = files.upload()

# Extract
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/kikuyu')

print('Files extracted:')
for root, dirs, files_list in os.walk('/content/kikuyu'):
    for f in files_list:
        print(' ', os.path.join(root, f))

In [ ]:
# Step 3 — Install Coqui TTS
!pip install -q TTS==0.22.0
print('TTS installed.')

In [ ]:
# Step 4 — Prepare dataset
import os, shutil, csv

CHUNKS_DIR = '/content/kikuyu/chunks'
DATASET_DIR = '/content/dataset'
AUDIO_DIR = os.path.join(DATASET_DIR, 'audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# Find metadata.csv
metadata_src = None
for root, dirs, files_list in os.walk('/content/kikuyu'):
    for f in files_list:
        if f == 'metadata.csv':
            metadata_src = os.path.join(root, f)

if not metadata_src:
    raise FileNotFoundError('metadata.csv not found in uploaded ZIP')

print(f'Found metadata: {metadata_src}')

# Copy WAVs and build flat metadata
flat_rows = []
missing = []

with open(metadata_src, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='|')
    for row in reader:
        filename = row['audio_file'].replace('chunks/', '')
        src = os.path.join(CHUNKS_DIR, filename)
        dst = os.path.join(AUDIO_DIR, filename)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            flat_rows.append({'audio_file': f'audio/{filename}', 'text': row['text'], 'speaker_name': 'speaker'})
        else:
            missing.append(filename)

# Write flat metadata
flat_meta = os.path.join(DATASET_DIR, 'metadata_train.csv')
with open(flat_meta, 'w', encoding='utf-8', newline='') as f:
    f.write('audio_file|text|speaker_name\n')
    for row in flat_rows:
        f.write(f"{row['audio_file']}|{row['text']}|{row['speaker_name']}\n")

print(f'Dataset ready: {len(flat_rows)} files')
if missing:
    print(f'Missing files ({len(missing)}): {missing[:5]}')

In [ ]:
# Step 5 — Find voice training WAV
speaker_wav = None
for root, dirs, files_list in os.walk('/content/kikuyu'):
    for f in files_list:
        if 'voice-training' in f or 'voice_training' in f:
            speaker_wav = os.path.join(root, f)

# Fallback: use first chunk
if not speaker_wav:
    wavs = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')]
    speaker_wav = wavs[0] if wavs else None

print(f'Speaker WAV: {speaker_wav}')

In [ ]:
# Step 6 — Fine-tune XTTS v2
from TTS.demos.xtts_ft_demo.utils.formatter import format_audio_list
from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get all speaker WAVs
all_wavs = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')]
print(f'Training with {len(all_wavs)} audio files...')

# Format audio for training
train_meta, eval_meta, audio_total_size = format_audio_list(
    all_wavs,
    target_language='en',
    out_path=OUTPUT_DIR,
    gradio_progress=None
)

print(f'Total audio: {audio_total_size:.1f}s')
print('Starting training...')

# Train
train_gpt(
    language='en',
    num_epochs=10,
    batch_size=4,
    grad_acumm=4,
    train_csv=train_meta,
    eval_csv=eval_meta,
    output_path=OUTPUT_DIR,
    max_audio_length=255995
)

print('\n✅ Training complete!')

In [ ]:
# Step 7 — Test the trained model
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import soundfile as sf
from IPython.display import Audio, display

# Load fine-tuned model
config = XttsConfig()
config.load_json(os.path.join(OUTPUT_DIR, 'config.json'))
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=OUTPUT_DIR, eval=True)
model.cuda()

# Test synthesis
test_text = 'Uhoro waku. Ndi mwega. Ningwendete.'
print(f'Synthesizing: {test_text}')

outputs = model.synthesize(
    test_text,
    config,
    speaker_wav=all_wavs[:5],
    language='en'
)

sf.write('/content/test_output.wav', outputs['wav'], 24000)
print('Playing test audio:')
display(Audio('/content/test_output.wav'))

In [ ]:
# Step 8 — Download the trained model
import shutil
from google.colab import files

print('Zipping model...')
shutil.make_archive('/content/kikuyu_model', 'zip', OUTPUT_DIR)

print('Downloading...')
files.download('/content/kikuyu_model.zip')
print('Done! Extract to coqui-server/output/ on your machine.')

## After downloading

1. Extract `kikuyu_model.zip` to `coqui-server/output/` on your machine
2. Update `coqui-server/main.py` — change the TTS model line from:
   ```python
   tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
   ```
   to:
   ```python
   tts = TTS(model_path='./output', config_path='./output/config.json').to(device)
   ```
3. Restart the server: `python main.py`

Your voice is now permanently baked into the model! 🎉